In [0]:
import requests
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

# 1. API Endpoint - We'll start with 'sessions' to see what races are happening
# This is live 2026 data!
URL = "https://api.openf1.org/v1/sessions?year=2026"

def ingest_f1_data():
    # Fetch data from API
    response = requests.get(URL)
    if response.status_code == 200:
        data = response.json()
        
        # Convert JSON to Spark DataFrame
        # We use Row(**d) to unpack the dictionary keys into columns
        raw_df = spark.createDataFrame([Row(**d) for d in data])
        
        # Add a 'ingestion_timestamp' so we know when the data was pulled
        # This is a standard professional practice
        bronze_df = raw_df.withColumn("ingested_at", current_timestamp())
        
        # Save to Bronze Table using Delta format
        (bronze_df.write
            .format("delta")
            .mode("overwrite") 
            .saveAsTable("f1_project.bronze.raw_sessions"))
            
        print(f"Success! Ingested {bronze_df.count()} sessions into Bronze.")
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}")

ingest_f1_data()

# Show the results
display(spark.table("f1_project.bronze.raw_sessions"))